# Bengali Toxicity — v7 (proven recipe + multi-seed ensemble)

Same recipe that scored **0.569** (BanglaBERT-base -> BiLSTM -> mean pool -> BCE, 5-fold), now **ensembled over 3 seeds** and averaged — the reliable way to gain ~+0.005-0.015. Reads official mount, submits test.csv IDs. ~1-1.5h on T4. **GPU + Internet ON, Run All.**


In [ ]:
# ---- 1. Install ----
import sys, subprocess
subprocess.run([sys.executable,"-m","pip","install","-q","iterative-stratification==0.1.7"], check=False)
print("deps done")

In [ ]:
# ---- 2. Imports & CONFIG ----
import os, re, gc, glob, random, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore")
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup, logging as hf_logging
from torch.optim import AdamW
hf_logging.set_verbosity_error()
class CFG:
    model_name = "csebuetnlp/banglabert"   # base (the proven winner). Do NOT switch to large.
    max_len    = 128
    n_folds    = 5
    epochs     = 3
    batch_size = 16
    lr         = 2e-5
    weight_decay = 0.01
    warmup_ratio = 0.1
    lstm_hidden  = 256
    dropout    = 0.3
    pos_weight_clip = 30.0
    GATE       = True
    seed       = 42
    use_fp16   = True
LABELS=["is_toxic","implicit_toxicity","sarcasm","contains_profanity","threatening","identity_attack","sexual_content"]
N_LABELS=len(LABELS)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG.seed)
print("Device:",DEVICE,"| Model:",CFG.model_name)

In [ ]:
# ---- 3. Metric helpers (Combined=(is_toxic macro-F1 + mean 6-attr F1)/2) ----
def gate_preds(prob,thr,gate=True):
    pred=(prob>=thr).astype(int)
    if gate: pred[pred[:,0]==0,1:]=0
    return pred
def combined_score(y,pred):
    A=f1_score(y[:,0],pred[:,0],average="macro",zero_division=0)
    B=np.mean([f1_score(y[:,i],pred[:,i],zero_division=0) for i in range(1,N_LABELS)])
    return (A+B)/2.0,A,B
def tune_thresholds(y,prob):
    grid=np.arange(0.10,0.90,0.01); thr=np.full(N_LABELS,0.5)
    bt,bs=0.5,-1
    for t in grid:
        s=f1_score(y[:,0],(prob[:,0]>=t).astype(int),average="macro",zero_division=0)
        if s>bs: bs,bt=s,t
    thr[0]=bt
    for i in range(1,N_LABELS):
        bt,bs=0.5,-1
        for t in grid:
            s=f1_score(y[:,i],(prob[:,i]>=t).astype(int),average="binary",zero_division=0)
            if s>bs: bs,bt=s,t
        thr[i]=bt if bs>0 else 0.5
    return thr

In [ ]:
# ---- 4. Load data (official mount; test.csv IDs) ----
def find_data_dir():
    for c in ["/kaggle/input/competitions/bengali-nlp-safety-challenge-2026","/kaggle/input/bengali-nlp-safety-challenge-2026"]:
        if os.path.exists(os.path.join(c,"train.csv")): return c
    hits=glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if hits: return os.path.dirname(sorted(hits,key=len)[0])
    for c in ["./data","."]:
        if os.path.exists(os.path.join(c,"train.csv")): return c
    raise FileNotFoundError("Attach the competition dataset.")
DATA_DIR=find_data_dir(); print("Data dir:",DATA_DIR)
train_df=pd.read_csv(os.path.join(DATA_DIR,"train.csv")); test_df=pd.read_csv(os.path.join(DATA_DIR,"test.csv"))
try: val_df=pd.read_csv(os.path.join(DATA_DIR,"validation.csv"))
except FileNotFoundError: val_df=None
def pick(df,pref,kw):
    for p in pref:
        if p in df.columns: return p
    for c in df.columns:
        if any(k in c.lower() for k in kw): return c
    o=[c for c in df.columns if df[c].dtype==object]; return max(o,key=lambda c: df[c].astype(str).str.len().mean())
TEXT_COL=pick(train_df,["text"],["text","comment"]); ID_COL=pick(test_df,["ID","id"],["id"])
full_df=pd.concat([train_df,val_df],ignore_index=True) if (val_df is not None and set(LABELS).issubset(val_df.columns)) else train_df.copy()
full_df=full_df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
for l in LABELS: full_df[l]=full_df[l].astype(int)
print("Full",full_df.shape,"Test",test_df.shape)

In [ ]:
# ---- 5. Cleaning (same as the winning notebook; no augmentation, no dedup) ----
def clean_text(t):
    if not isinstance(t,str): return ""
    t=re.sub(r"http\S+|www\.\S+","",t)
    t=re.sub(r"<.*?>","",t)
    t=re.sub(r"@\w+","",t)
    t=re.sub(r"\s+"," ",t).strip()
    t=t.replace("#","")
    return t
full_df["clean"]=full_df[TEXT_COL].apply(clean_text)
test_df["clean"]=test_df[TEXT_COL].apply(clean_text)
print(full_df[["clean"]].head(3).to_string())

In [ ]:
# ---- 6. Tokenizer, Dataset, Model (BanglaBERT -> BiLSTM -> mean pool), pos_weight ----
tokenizer=AutoTokenizer.from_pretrained(CFG.model_name)
class DS(Dataset):
    def __init__(s,texts,labels=None): s.t=list(texts); s.y=labels
    def __len__(s): return len(s.t)
    def __getitem__(s,i):
        e=tokenizer(str(s.t[i]),add_special_tokens=True,max_length=CFG.max_len,padding="max_length",
                    truncation=True,return_token_type_ids=False,return_attention_mask=True,return_tensors="pt")
        it={"input_ids":e["input_ids"].flatten(),"attention_mask":e["attention_mask"].flatten()}
        if s.y is not None: it["labels"]=torch.tensor(s.y[i],dtype=torch.float)
        return it
class BertBiLSTM(nn.Module):
    def __init__(s,name,n_out=7):
        super().__init__()
        s.bert=AutoModel.from_pretrained(name)
        s.lstm=nn.LSTM(s.bert.config.hidden_size,CFG.lstm_hidden,num_layers=2,batch_first=True,bidirectional=True,dropout=CFG.dropout)
        s.dropout=nn.Dropout(CFG.dropout)
        s.fc=nn.Linear(CFG.lstm_hidden*2,n_out)
    def forward(s,input_ids,attention_mask):
        seq=s.bert(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        lo,_=s.lstm(seq)
        # masked mean pool over real tokens
        m=attention_mask.unsqueeze(-1).float()
        pooled=(lo*m).sum(1)/m.sum(1).clamp(min=1e-9)
        return s.fc(s.dropout(pooled))
def pos_weight_from(y):
    pos=y.sum(0); neg=len(y)-pos
    return np.clip(neg/np.clip(pos,1,None),1.0,CFG.pos_weight_clip)
print("model ready")

In [ ]:
# ---- 7. Train one fold ----
@torch.no_grad()
def predict(model,loader):
    model.eval(); out=[]
    for b in loader:
        ids=b["input_ids"].to(DEVICE); mask=b["attention_mask"].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CFG.use_fp16): lg=model(ids,mask)
        out.append(torch.sigmoid(lg).float().cpu().numpy())
    return np.concatenate(out,0)
def train_fold(trt,trY,vat,vaY,tet,fold):
    seed_everything(CFG.seed+fold)
    model=BertBiLSTM(CFG.model_name,N_LABELS).to(DEVICE)
    trl=DataLoader(DS(trt,trY),batch_size=CFG.batch_size,shuffle=True,num_workers=0,pin_memory=True)
    vl=DataLoader(DS(vat,vaY),batch_size=64,shuffle=False,num_workers=0)
    el=DataLoader(DS(tet),batch_size=64,shuffle=False,num_workers=0)
    pw=torch.tensor(pos_weight_from(trY),dtype=torch.float,device=DEVICE)
    crit=nn.BCEWithLogitsLoss(pos_weight=pw)
    opt=AdamW(model.parameters(),lr=CFG.lr,weight_decay=CFG.weight_decay)
    steps=len(trl)*CFG.epochs
    sch=get_linear_schedule_with_warmup(opt,int(steps*CFG.warmup_ratio),steps)
    scaler=torch.cuda.amp.GradScaler(enabled=CFG.use_fp16)
    best,bv,bt=-1,None,None
    for ep in range(CFG.epochs):
        model.train(); run=0
        for b in trl:
            ids=b["input_ids"].to(DEVICE); mask=b["attention_mask"].to(DEVICE); y=b["labels"].to(DEVICE)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=CFG.use_fp16): loss=crit(model(ids,mask),y)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update(); sch.step(); run+=loss.item()
        vp=predict(model,vl); th=tune_thresholds(vaY,vp)
        c,A,B=combined_score(vaY,gate_preds(vp,th,CFG.GATE))
        print(f"  fold{fold} ep{ep+1}/{CFG.epochs} loss={run/len(trl):.4f} valCombined={c:.4f} (A={A:.3f} B={B:.3f})")
        if c>=best: best=c; bv=vp; bt=predict(model,el)
    del model; gc.collect(); torch.cuda.empty_cache()
    return bv,bt

In [ ]:
# ---- 8. Multi-seed 5-fold CV (ensemble of seeds — the proven booster) ----
SEEDS=[42, 1337, 2024]   # 3 seeds x 5 folds = 15 base models (~1-1.5h). Drop to [42,1337] if quota tight.
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
    folds=list(MultilabelStratifiedKFold(n_splits=CFG.n_folds,shuffle=True,random_state=42).split(full_df["clean"].values,full_df[LABELS].values)); print("MLStratKFold")
except Exception as e:
    from sklearn.model_selection import StratifiedKFold
    folds=list(StratifiedKFold(n_splits=CFG.n_folds,shuffle=True,random_state=42).split(full_df["clean"].values,full_df["is_toxic"].values)); print("StratKFold",e)
X=full_df["clean"].values; Y=full_df[LABELS].values.astype(float); test_texts=test_df["clean"].values
oof=np.zeros((len(full_df),N_LABELS)); test_pred=np.zeros((len(test_df),N_LABELS))
for seed in SEEDS:
    CFG.seed=seed
    print(f"########## SEED {seed} ##########")
    for fold,(tr,va) in enumerate(folds):
        print(f"===== seed {seed} fold {fold+1}/{CFG.n_folds} =====")
        vp,tp=train_fold(X[tr],Y[tr],X[va],Y[va],test_texts,fold)
        oof[va]+=vp/len(SEEDS); test_pred+=tp/(CFG.n_folds*len(SEEDS))
np.save("oof.npy",oof); np.save("test_pred.npy",test_pred)
thr=tune_thresholds(Y,oof); op=gate_preds(oof,thr,CFG.GATE); c,A,B=combined_score(Y,op)
print("="*54)
print(f"OOF Subtask A={A:.4f} | Subtask B={B:.4f}")
print(f"OOF COMBINED SCORE = {c:.4f}   <-- expected leaderboard")
print("="*54)
for i,l in enumerate(LABELS):
    ff=f1_score(Y[:,i],op[:,i],average=("macro" if i==0 else "binary"),zero_division=0)
    print(f"  {l:<20} thr={thr[i]:.2f} F1={ff:.4f}")

In [ ]:
# ---- 9. Submission (test.csv IDs) ----
final=gate_preds(test_pred,thr,CFG.GATE)
sub=pd.DataFrame({ID_COL:test_df[ID_COL].values})
for i,l in enumerate(LABELS): sub[l]=final[:,i]
sub.to_csv("submission.csv",index=False)
print("submission.csv",sub.shape,"| IDs from official test.csv"); print(sub.head())
print("positive rate:"); print(sub[LABELS].mean().round(3))